In [ ]:
%%sql

-- ============================================================
-- GOLD FIRE EVOLUTION
-- ============================================================
-- Granularidad:
--   1 fila = 1 detección NASA
--
-- Objetivo:
--   Mantener el máximo nivel de detalle de las detecciones
--   de incendios para permitir posteriormente en Power BI:
--
--     - evolución temporal
--     - mapas
--     - filtros por fecha/hora
--     - análisis de intensidad
--     - análisis espacial
--     - agregaciones por zona
--
-- La agregación NO se realiza aquí.
-- Power BI puede agregar posteriormente por:
--     hora, día, zona, intensidad, etc.
--
-- Clave técnica:
--     fire_id
--
-- fire_id se construye de forma determinista a partir de:
--     latitude + longitude + fire_detection_timestamp
--
-- ============================================================


-- ------------------------------------------------------------
-- 1. Crear tabla Gold
-- ------------------------------------------------------------

CREATE TABLE IF NOT EXISTS gold_fire_evolution (

    fire_id STRING,

    latitude DOUBLE,
    longitude DOUBLE,

    acq_date DATE,
    acq_time INT,

    fire_detection_timestamp TIMESTAMP,

    bright_ti4 DOUBLE,
    bright_ti5 DOUBLE,

    fire_radiative_power DOUBLE,

    confidence STRING,
    daynight STRING,

    -- --------------------------------------------------------
    -- Dimensiones temporales derivadas
    -- --------------------------------------------------------

    detection_date DATE,
    detection_hour INT,
    detection_minute INT,

    detection_year INT,
    detection_month INT,
    detection_day INT,

    -- --------------------------------------------------------
    -- Indicador de intensidad
    -- --------------------------------------------------------

    fire_intensity_level STRING,

    -- --------------------------------------------------------
    -- Trazabilidad
    -- --------------------------------------------------------

    landing_source_file STRING,
    ingestion_timestamp TIMESTAMP,

    updated_source_file STRING,
    updated_timestamp TIMESTAMP,

    gold_updated_at TIMESTAMP
);


-- ------------------------------------------------------------
-- 2. Preparar fuente incremental
-- ------------------------------------------------------------
--
-- No utilizamos una ventana temporal artificial aquí.
--
-- Silver ya contiene las detecciones disponibles.
-- El MERGE evita duplicados y permite actualizar registros
-- si los datos de origen cambian.
--
-- ------------------------------------------------------------

CREATE OR REPLACE TEMP VIEW fire_evolution_source AS

SELECT

    -- --------------------------------------------------------
    -- Identificador técnico determinista
    -- --------------------------------------------------------

    SHA2(
        CONCAT_WS(
            '|',
            CAST(latitude AS STRING),
            CAST(longitude AS STRING),
            CAST(fire_detection_timestamp AS STRING)
        ),
        256
    ) AS fire_id,

    -- --------------------------------------------------------
    -- Datos originales NASA
    -- --------------------------------------------------------

    CAST(latitude AS DOUBLE) AS latitude,

    CAST(longitude AS DOUBLE) AS longitude,

    acq_date,

    acq_time,

    fire_detection_timestamp,

    CAST(bright_ti4 AS DOUBLE) AS bright_ti4,

    CAST(bright_ti5 AS DOUBLE) AS bright_ti5,

    CAST(fire_radiative_power AS DOUBLE)
        AS fire_radiative_power,

    confidence,

    daynight,

    -- --------------------------------------------------------
    -- Dimensiones temporales
    -- --------------------------------------------------------

    CAST(
        DATE(fire_detection_timestamp)
        AS DATE
    ) AS detection_date,

    HOUR(fire_detection_timestamp)
        AS detection_hour,

    MINUTE(fire_detection_timestamp)
        AS detection_minute,

    YEAR(fire_detection_timestamp)
        AS detection_year,

    MONTH(fire_detection_timestamp)
        AS detection_month,

    DAY(fire_detection_timestamp)
        AS detection_day,

    -- --------------------------------------------------------
    -- Nivel de intensidad basado en FRP
    --
    -- Es un indicador analítico, no una clasificación
    -- científica del incendio.
    -- --------------------------------------------------------

    CASE

        WHEN fire_radiative_power >= 100
            THEN 'VERY_HIGH'

        WHEN fire_radiative_power >= 50
            THEN 'HIGH'

        WHEN fire_radiative_power >= 10
            THEN 'MEDIUM'

        WHEN fire_radiative_power >= 0
            THEN 'LOW'

        ELSE 'UNKNOWN'

    END AS fire_intensity_level,

    -- --------------------------------------------------------
    -- Trazabilidad
    -- --------------------------------------------------------

    landing_source_file,

    ingestion_timestamp,

    updated_source_file,

    updated_timestamp,

    current_timestamp() AS gold_updated_at

FROM silver_nasa_fires

WHERE fire_detection_timestamp IS NOT NULL
  AND latitude IS NOT NULL
  AND longitude IS NOT NULL;


-- ------------------------------------------------------------
-- 3. Eliminar posibles duplicados dentro de la propia fuente
-- ------------------------------------------------------------
--
-- El MERGE de Delta no puede tener múltiples filas de source
-- que coincidan con el mismo target.
--
-- En caso de existir varias versiones del mismo registro,
-- conservamos la más recientemente ingerida/actualizada.
-- ------------------------------------------------------------

CREATE OR REPLACE TEMP VIEW fire_evolution_deduplicated AS

SELECT *

FROM (

    SELECT

        fes.*,

        ROW_NUMBER() OVER (

            PARTITION BY fire_id

            ORDER BY

                COALESCE(
                    updated_timestamp,
                    ingestion_timestamp
                ) DESC

        ) AS rn

    FROM fire_evolution_source fes

)

WHERE rn = 1;


-- ------------------------------------------------------------
-- 4. MERGE incremental
-- ------------------------------------------------------------
--
-- Si la detección ya existe:
--     UPDATE
--
-- Si aparece una detección nueva:
--     INSERT
--
-- Esto permite ejecutar el proceso cada 30 minutos.
-- ------------------------------------------------------------

MERGE INTO gold_fire_evolution AS target

USING fire_evolution_deduplicated AS source

ON target.fire_id = source.fire_id


WHEN MATCHED THEN

    UPDATE SET

        target.latitude =
            source.latitude,

        target.longitude =
            source.longitude,

        target.acq_date =
            source.acq_date,

        target.acq_time =
            source.acq_time,

        target.fire_detection_timestamp =
            source.fire_detection_timestamp,

        target.bright_ti4 =
            source.bright_ti4,

        target.bright_ti5 =
            source.bright_ti5,

        target.fire_radiative_power =
            source.fire_radiative_power,

        target.confidence =
            source.confidence,

        target.daynight =
            source.daynight,

        target.detection_date =
            source.detection_date,

        target.detection_hour =
            source.detection_hour,

        target.detection_minute =
            source.detection_minute,

        target.detection_year =
            source.detection_year,

        target.detection_month =
            source.detection_month,

        target.detection_day =
            source.detection_day,

        target.fire_intensity_level =
            source.fire_intensity_level,

        target.landing_source_file =
            source.landing_source_file,

        target.ingestion_timestamp =
            source.ingestion_timestamp,

        target.updated_source_file =
            source.updated_source_file,

        target.updated_timestamp =
            source.updated_timestamp,

        target.gold_updated_at =
            source.gold_updated_at


WHEN NOT MATCHED THEN

    INSERT (

        fire_id,

        latitude,
        longitude,

        acq_date,
        acq_time,

        fire_detection_timestamp,

        bright_ti4,
        bright_ti5,

        fire_radiative_power,

        confidence,
        daynight,

        detection_date,
        detection_hour,
        detection_minute,

        detection_year,
        detection_month,
        detection_day,

        fire_intensity_level,

        landing_source_file,
        ingestion_timestamp,

        updated_source_file,
        updated_timestamp,

        gold_updated_at

    )

    VALUES (

        source.fire_id,

        source.latitude,
        source.longitude,

        source.acq_date,
        source.acq_time,

        source.fire_detection_timestamp,

        source.bright_ti4,
        source.bright_ti5,

        source.fire_radiative_power,

        source.confidence,
        source.daynight,

        source.detection_date,
        source.detection_hour,
        source.detection_minute,

        source.detection_year,
        source.detection_month,
        source.detection_day,

        source.fire_intensity_level,

        source.landing_source_file,
        source.ingestion_timestamp,

        source.updated_source_file,
        source.updated_timestamp,

        source.gold_updated_at

    );